In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
%config Completer.use_jedi = False

In [2]:
clin = pd.read_csv('clin_ready.csv', header = 0, index_col=0)
prot = pd.read_csv('raw_data_proteins_filtered.csv',header = 0, index_col = 0)
prot_list = pd.read_csv('prot_filtered.csv', header = 0, index_col=0)

prot = prot.filter(prot_list.iloc[:,0], axis = 1)
prot = prot.drop(labels = [1578], axis = 0)
clin = clin.drop(labels = [1578], axis = 0)

print("PROT SHAPE:", prot.shape)
print("CLIN SHAPE:", clin.shape)

assert prot.shape[0] == clin.shape[0]

PROT SHAPE: (414, 4979)
CLIN SHAPE: (414, 38)


In [4]:
# get index of patients in given iss range
def get_index(min_, max_):
    index=[]
    for i in clin.index:
        if clin.loc[i,'iss']>=min_ and clin.loc[i,'iss']<=max_:
            index.append(i)
    print("n. of patients:", len(index))
    return index

# filter the clin df with these indexes and get stats
def return_df(index_list):
    filtered_df = clin.filter(items = index_list, axis = 0)
    filtered_df = filtered_df.drop(columns=['injury_time',
            '24hr_crystalloid_volume', 
            '24hr_colloid_volume_ml',
            '24hr_prbc_volume', 
            '24hr_ffp_volume',
            '24hr_plts_volume', 
            '24hr_cryoprecipitate_volume',
            '24hr_octaplas_volume',
            '24hr_hypertonic_saline_volume',
            'injury_to_death',
            'vent_free_calculated'], axis = 1)
                                   
    info = filtered_df.describe()
    for col in info.columns:
        info[col]=info[col].round(1)
    info.drop(index = ['count', 'min', '25%', '50%', '75%', 'max'], inplace=True)                           
                                   
    return info

# join mean and std
def join_mean_std(dataset, dataset_name):
    joined=[]
    for i in dataset.T.index:
        joined.append(' ± '.join(dataset.T.loc[i,:].astype(str)))
    joined_ = pd.DataFrame(joined)
    joined_ = joined_.T
    joined_.columns = dataset.columns
    joined_.index =[dataset_name]
    return joined_ 

In [5]:
# groups
index_low = get_index(0, 3)
index_medium = get_index(4, 8)
index_moderate = get_index(9, 15)
index_severe = get_index(16, 24)
index_critical = get_index(25, 50)

# return df
df_low = return_df(index_low)
df_medium = return_df(index_medium)
df_moderate = return_df(index_moderate)
df_severe = return_df(index_severe)
df_critical = return_df(index_critical)

n. of patients: 73
n. of patients: 77
n. of patients: 79
n. of patients: 77
n. of patients: 108


In [6]:
# join mean and std
Low = join_mean_std(df_low, "Control")
Medium = join_mean_std(df_medium, "Mild")
Moderate = join_mean_std(df_moderate, "Moderate")
Severe = join_mean_std(df_severe, "Severe")
Critical = join_mean_std(df_critical, "Critical")

# join all tables
full_dem_table = Low.append([Medium, Moderate,Severe, Critical])

In [7]:
full_dem_table

,age,iss,admission_hr,admission_sbp,admission_gcs,injury_to_baseline_sample_time,baseline_hb,baseline_hct,baseline_wcc,baseline_plt,baseline_lactate,baseline_base_deficit,itu_days,hdu_days,ventilator_days,ventilator_free_days,length_of_stay,28_day_mortality,percent_vent_free,total_txa
Control,38.5 ± 16.4,1.0 ± 0.6,88.5 ± 22.1,137.6 ± 25.9,13.7 ± 2.5,86.0 ± 20.3,14.4 ± 1.3,0.4 ± 0.0,10.0 ± 4.0,232.7 ± 56.5,2.3 ± 2.4,-0.1 ± 3.3,0.0 ± 0.3,0.0 ± 0.1,0.0 ± 0.1,27.9 ± 0.2,1.7 ± 2.3,1.0 ± 0.0,69.7 ± 46.1,0.0 ± 0.2
Mild,38.9 ± 14.8,5.1 ± 1.4,84.8 ± 18.9,142.2 ± 25.1,12.8 ± 3.4,84.4 ± 20.4,16.1 ± 15.0,0.4 ± 0.0,10.8 ± 4.1,222.3 ± 57.2,2.1 ± 1.8,0.4 ± 2.8,1.0 ± 4.2,0.1 ± 0.4,0.2 ± 1.2,27.5 ± 2.5,7.0 ± 9.4,1.0 ± 0.0,90.0 ± 29.1,0.1 ± 0.4
Moderate,41.7 ± 17.4,10.3 ± 1.8,84.8 ± 20.2,133.9 ± 24.5,13.0 ± 3.6,87.3 ± 22.1,13.9 ± 1.7,0.4 ± 0.0,12.7 ± 5.7,228.8 ± 70.2,2.6 ± 2.3,0.5 ± 3.1,1.6 ± 6.8,0.8 ± 3.5,0.3 ± 1.3,27.2 ± 2.8,15.5 ± 16.8,1.0 ± 0.2,94.5 ± 20.2,0.3 ± 0.7
Severe,43.2 ± 18.6,19.4 ± 2.5,88.8 ± 20.7,134.5 ± 29.4,11.8 ± 4.2,88.4 ± 26.5,13.9 ± 1.9,0.4 ± 0.1,16.2 ± 6.2,229.2 ± 56.9,2.5 ± 2.4,1.8 ± 3.6,3.0 ± 6.8,1.4 ± 3.6,1.1 ± 2.9,25.7 ± 5.4,18.2 ± 15.7,0.9 ± 0.2,92.0 ± 22.8,0.6 ± 0.9
Critical,45.3 ± 18.0,33.9 ± 6.8,101.3 ± 30.0,124.4 ± 33.6,10.8 ± 4.3,96.5 ± 22.8,13.2 ± 2.1,0.4 ± 0.1,17.1 ± 7.2,214.4 ± 64.5,3.1 ± 2.6,3.4 ± 6.3,7.8 ± 10.6,1.1 ± 3.3,2.4 ± 3.7,23.4 ± 6.8,26.7 ± 27.4,0.9 ± 0.4,80.3 ± 31.6,0.4 ± 0.6


In [11]:
mods = pd.read_csv('/Users/smasarone/Downloads/New.MODS.groups(2022).csv', index_col=0, header=0)
mods['MODS.group'].unique()

# why 411 and not 414 because 3 patients didn't go to ICU

array(['NO MODS', 'Short MODS', 'Prolonged MODS', 'Early Death',
       'Late Death'], dtype=object)

In [12]:
mods['MODS.group'].filter(index_critical, axis =0).value_counts()

NO MODS           49
Short MODS        28
Prolonged MODS    15
Early Death       12
Late Death         4
Name: MODS.group, dtype: int64

In [14]:
# read new length of stay data
new_los = pd.read_csv('/Users/smasarone/Downloads/clinical_data_JR_20211110.csv', index_col = 0)
clin['NEW_LOS'] = new_los['length_of_stay_new']

# low
def filter_and_get_above14(index):
    dataset = clin.filter(items = index, axis =0)
    counter=0
    for i in dataset['NEW_LOS']:
        if i>14:
            counter+=1
    return counter

print(filter_and_get_above14(index_low))
print(filter_and_get_above14(index_medium))
print(filter_and_get_above14(index_moderate))
print(filter_and_get_above14(index_severe))
print(filter_and_get_above14(index_critical))

2
11
30
37
66


In [15]:
full_dem_table.rename(columns={'iss':'iss_mean'})
full_dem_table['iss_range']=['0-3', '4-8', '9-15', '16-24', '25-50']

# full_dem_table.index = ['Low', 'Medium', 'Moderate', 'Severe', 'Critical']
range_df = pd.DataFrame(full_dem_table['iss_range'], index = full_dem_table.index)
range_df['patients_count']= [73, 77, 79, 77, 108]
range_df['injury_to_baseline_time']= full_dem_table['injury_to_baseline_sample_time']
range_df = range_df.join(full_dem_table.iloc[:,1:20])
range_df=range_df.drop(columns = ['iss',
                                  'baseline_hb',
                                  'baseline_hct',
                                  'baseline_plt',
                                  'itu_days',
                                  'hdu_days',
                                  'admission_gcs',
                                  'admission_hr',
                                  'ventilator_days',
                                  'percent_vent_free',
                                  'total_txa', 
                                  'ventilator_free_days', 
                                 '28_day_mortality', 
                                 'injury_to_baseline_sample_time'])

indices=[index_low, index_medium, index_moderate, index_severe, index_critical]
#above_14 = los_above_14d(indices)
# range_df['Lenght of stay above 14d'] = above_14

# mortality, inf, MODS has to be added as n of adv outcome over cases
deaths = ['0/73 (0)','0/77 (0)', '2/79 (0.03)', '5/77 (0.06)','16/108 (0.15)']
range_df['Mortality_rate'] = deaths

infections=['1/65 (0.02)','5/71 (0.07)','14/71 (0.2)','14/61 (0.23)','32/73 (0.44)']
range_df['Length of stay >14d'] = [2, 11, 30, 37, 66]
range_df['No Mods']=[72, 72, 66, 55, 49]
range_df['Short Mods']=[1, 4, 8, 12, 28]
range_df['Prolonged Mods']=[0, 1, 2, 3, 15]
range_df['Early Death']=[0, 0, 1, 4, 12]
range_df['Late Death']= [0, 0, 1, 1, 4]

range_df.columns = ['Iss range', 'Patients count', 'Injury to baseline time',
       'Systolic Blood pressure (<120mmHg)', 'Baseline wcc (4.5-11 10*9/L)', 'Baseline lactate (0.5-2.2mmol/L)',
       'Baseline base deficit(-2/+2 mEq/L)', 'Length of stay','Mortality rate',
       'Length of stay >14d', 'No Mods','Short Mods','Prolonged Mods', 'Early Death', 'Late Death']

In [16]:
# Note that this MODS does not include patients who died
pd.options.display.latex.repr = True
range_df

,Iss range,Patients count,Injury to baseline time,Systolic Blood pressure (<120mmHg),Baseline wcc (4.5-11 10*9/L),Baseline lactate (0.5-2.2mmol/L),Baseline base deficit(-2/+2 mEq/L),Length of stay,Mortality rate,Length of stay >14d,No Mods,Short Mods,Prolonged Mods,Early Death,Late Death
Control,0-3,73,86.0 ± 20.3,137.6 ± 25.9,10.0 ± 4.0,2.3 ± 2.4,-0.1 ± 3.3,1.7 ± 2.3,0/73 (0),2,72,1,0,0,0
Mild,4-8,77,84.4 ± 20.4,142.2 ± 25.1,10.8 ± 4.1,2.1 ± 1.8,0.4 ± 2.8,7.0 ± 9.4,0/77 (0),11,72,4,1,0,0
Moderate,9-15,79,87.3 ± 22.1,133.9 ± 24.5,12.7 ± 5.7,2.6 ± 2.3,0.5 ± 3.1,15.5 ± 16.8,2/79 (0.03),30,66,8,2,1,1
Severe,16-24,77,88.4 ± 26.5,134.5 ± 29.4,16.2 ± 6.2,2.5 ± 2.4,1.8 ± 3.6,18.2 ± 15.7,5/77 (0.06),37,55,12,3,4,1
Critical,25-50,108,96.5 ± 22.8,124.4 ± 33.6,17.1 ± 7.2,3.1 ± 2.6,3.4 ± 6.3,26.7 ± 27.4,16/108 (0.15),66,49,28,15,12,4


In [20]:
# export tables as png (pdf doesn't seem to work)
import dataframe_image as dfi
dfi.export(range_df, 'dataframe.png')